# 01 - الإعدادات وإنشاء العينة الصغيرة

هذا النوتبوك يغطي:
1. تحميل الإعدادات المركزية من `config/settings.py`
2. فحص الاتصال بـ MongoDB (اختياري - لن يوقف التنفيذ إن لم يكن متاحًا الآن)
3. تجربة سكربت `create_small_sample.py` بمجرد توفر الملف الضخم داخل `data/`

> ملاحظة: الملف `orders_huge_mixed_quality.csv` لم يُرفع بعد. الخلايا مكتوبة بحيث تعمل بأمان الآن، وتُنتج العينة تلقائيًا فور وضع الملف داخل مجلد `data/`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> جذر المشروع
sys.path.append(str(PROJECT_ROOT))

from config import settings

settings.ensure_directories()

print(f"BASE_DIR              : {settings.BASE_DIR}")
print(f"SOURCE_FILE_PATH       : {settings.SOURCE_FILE_PATH}")
print(f"SAMPLE_FILE_PATH       : {settings.SAMPLE_FILE_PATH}")
print(f"SMALL_FILE_THRESHOLD_MB: {settings.SMALL_FILE_THRESHOLD_MB}")
print(f"BATCH_SIZE             : {settings.BATCH_SIZE}")
print(f"MONGO_URI              : {settings.MONGO_URI}")
print(f"MONGO_DB_NAME          : {settings.MONGO_DB_NAME}")

BASE_DIR              : D:\midterm-data-pipeline
SOURCE_FILE_PATH       : D:\midterm-data-pipeline\data\orders_huge_mixed_quality.csv
SAMPLE_FILE_PATH       : D:\midterm-data-pipeline\data\orders_small_sample.csv
SMALL_FILE_THRESHOLD_MB: 200.0
BATCH_SIZE             : 5000
MONGO_URI              : mongodb://localhost:27017
MONGO_DB_NAME          : midterm_pipeline_db


## فحص الاتصال بـ MongoDB

نتأكد من أن MongoDB يعمل ويمكن الوصول إليه قبل البدء بأي تحميل لاحقًا. إذا لم يكن متاحًا الآن، الخلية تطبع تحذيرًا فقط ولا توقف باقي النوتبوك.

In [2]:
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError

mongo_available = False
try:
    client = MongoClient(settings.MONGO_URI, serverSelectionTimeoutMS=3000)
    client.admin.command("ping")
    db = client[settings.MONGO_DB_NAME]
    mongo_available = True
    print(f"[OK] متصل بـ MongoDB على {settings.MONGO_URI}")
    print(f"[OK] قاعدة البيانات المستهدفة: {settings.MONGO_DB_NAME}")
    print(f"Collections الحالية: {db.list_collection_names()}")
except ServerSelectionTimeoutError as e:
    print("[تحذير] لا يمكن الاتصال بـ MongoDB الآن.")
    print("تأكد أن الخدمة تعمل (mongod) أو أن MONGO_URI صحيح في البيئة.")
    print(f"تفاصيل الخطأ: {e}")

[OK] متصل بـ MongoDB على mongodb://localhost:27017
[OK] قاعدة البيانات المستهدفة: midterm_pipeline_db
Collections الحالية: []


## التحقق من وجود الملف الضخم المصدر

بمجرد ما ترفع `orders_huge_mixed_quality.csv` إلى مجلد `data/`، هذه الخلية بتكتشفه تلقائيًا وتطبع حجمه.

In [5]:
source_exists = settings.SOURCE_FILE_PATH.exists()

if source_exists:
    size_mb = settings.SOURCE_FILE_PATH.stat().st_size / (1024 * 1024)
    print(f"[OK] الملف موجود: {settings.SOURCE_FILE_PATH}")
    print(f"الحجم: {size_mb:.2f} MB")
else:
    print(f"[لم يُرفع بعد] الملف غير موجود في: {settings.SOURCE_FILE_PATH}")
    print("ضع الملف في مجلد data/ بنفس الاسم، ثم أعد تشغيل هذه الخلية.")

[OK] الملف موجود: D:\midterm-data-pipeline\data\orders_huge_mixed_quality.csv
الحجم: 12650.32 MB


## إنشاء العينة الصغيرة (create_small_sample)

نستدعي الدالة مباشرة من `src/create_small_sample.py` (نفس المنطق المستخدم في سطر الأوامر):

```bash
python src/create_small_sample.py --input data/orders_huge_mixed_quality.csv --rows 100000
```

الخلية التالية تعمل تلقائيًا فقط إذا كان الملف المصدر موجودًا.

In [6]:
sys.path.append(str(PROJECT_ROOT / "src"))
from create_small_sample import create_small_sample

SAMPLE_ROWS = 100_000  # قابل للتغيير من هنا أو تمريره كمعامل

if source_exists:
    result = create_small_sample(
        input_path=settings.SOURCE_FILE_PATH,
        output_path=settings.SAMPLE_FILE_PATH,
        rows=SAMPLE_ROWS,
        mode="head",   # أو "random" لعينة موزعة عبر كامل الملف
        seed=42,
    )
    print("== نتيجة إنشاء العينة ==")
    for k, v in result.items():
        print(f"{k}: {v}")
else:
    print("تم تخطي هذه الخطوة: الملف المصدر غير موجود بعد.")

== نتيجة إنشاء العينة ==
mode: head
rows_written: 100000
input_path: D:\midterm-data-pipeline\data\orders_huge_mixed_quality.csv
output_path: D:\midterm-data-pipeline\data\orders_small_sample.csv
elapsed_seconds: 0.999
input_size_mb: 12650.32
output_size_mb: 41.765


## الخطوة القادمة

بعد رفع الملف الضخم وتشغيل هذا النوتبوك بنجاح (يظهر ملف `orders_small_sample.csv` داخل `data/`)، ننتقل إلى:

**`02_file_router.ipynb`** — بناء نقطة التشغيل الموحدة (Router) التي تفحص حجم الملف وتختار المحرك المناسب تلقائيًا (Python Batch أو PySpark).